<a href="https://colab.research.google.com/github/swetasm108-bit/Grammar-Scoring-Engine/blob/main/Grammar_Scoring_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
!pip install -q resampy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 59.3 MB/s eta 0:00:00


In [ ]:
# Clone the project from GitHub
!git clone https://github.com/sm6746/SHL.git

# Move into the project directory
%cd SHL

Cloning into 'SHL'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 22 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 19.12 KiB | 3.19 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/SHL


In [ ]:
!pip install openai-whisper gingerit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 35.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 7.0 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=164acd3b28efab8dcdbd3a9a4675b37c575acbe931b8e025b93f476d7796f4d9
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
  Created wheel for gingerit: filename=gingerit-0.0.0.1-py3-none-any.whl size=1305 sha256=120451df1b4992edeafce8c10bbf5657fd4ac41710844ce61a6f50a3761b9838
  Stored in directory: /root/.cache/pip/wheels/23/a7/02/2ea6493213411d9a392886e97d43684febd4fbd3d5519af183
Successfully built openai-whisper gingerit


In [ ]:
# Download the shl-sarah-project dataset
!kaggle datasets download -d sarahkhambatta/shl-sarah-project

# Unzip the files into a folder named 'shl_data'
!mkdir -p shl_data
!unzip -q shl-sarah-project.zip -d shl_data

# List files to verify
!ls shl_data

Dataset URL: https://www.kaggle.com/datasets/sarahkhambatta/shl-sarah-project
License(s): unknown
  0% 0.00/6.52k [00:00<?, ?B/s]
100% 6.52k/6.52k [00:00<00:00, 14.2MB/s]
app.py		README.md	  sample_submission.csv  train.csv
notebook.ipynb	requirements.txt  test.csv


In [ ]:
import os
import pandas as pd

BASE_DIR = '/content/sample_data'

print("Files inside base directory:")
print(os.listdir(BASE_DIR))

train_df = pd.read_csv(os.path.join(BASE_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(BASE_DIR, 'test.csv'))
print(train_df.head(),test_df.head())

Files inside base directory:
['anscombe.json', 'README.md', 'test.csv', 'train.csv', 'mnist_train_small.csv', 'california_housing_test.csv', 'mnist_test.csv', 'california_housing_train.csv']
         filename  label
0  audio_1261.wav    1.0
1   audio_942.wav    1.5
2  audio_1110.wav    1.5
3  audio_1024.wav    1.5
4   audio_538.wav    2.0          filename
0   audio_706.wav
1   audio_800.wav
2    audio_68.wav
3  audio_1267.wav
4   audio_683.wav


In [43]:
import pandas as pd
import numpy as np
import librosa
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline


np.random.seed(42)
tf.random.set_seed(42)

In [44]:
def extract_audio_features(file_path):
    try:
        # Load audio file
        audio, sample_rate = librosa.load(file_path, res_type='kaiser_fast')

        # Extract MFCCs (standard is 40 features)
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
        mfccs_scaled = np.mean(mfccs.T, axis=0)

        return mfccs_scaled
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

In [45]:
import os
import librosa
import numpy as np



X_train = []
y_train = []
for idx, row in train_df.iterrows():
    audio_path = os.path.join(TRAIN_AUDIO_DIR, row['filename'])
    features = extract_features(audio_path)
    if features is not None:
        X_train.append(features)
        y_train.append(row['label'])

X_train = np.array(X_train)
y_train = np.array(y_train)
print(f"Processed {len(X_train)} training samples")


X_test = []
test_filenames = []
for idx, row in test_df.iterrows():
    audio_path = os.path.join(TEST_AUDIO_DIR, row['filename'])
    features = extract_features(audio_path)
    if features is not None:
        X_test.append(features)
        test_filenames.append(row['filename'])

X_test = np.array(X_test)
print(f"Processed {len(X_test)} test samples")

Processed 444 training samples
Error processing /content/dataset/test_data/test_data/audio_706.wav: [Errno 2] No such file or directory: '/content/dataset/test_data/test_data/audio_706.wav'
Error processing /content/dataset/test_data/test_data/audio_800.wav: [Errno 2] No such file or directory: '/content/dataset/test_data/test_data/audio_800.wav'
Error processing /content/dataset/test_data/test_data/audio_68.wav: [Errno 2] No such file or directory: '/content/dataset/test_data/test_data/audio_68.wav'
Error processing /content/dataset/test_data/test_data/audio_1267.wav: [Errno 2] No such file or directory: '/content/dataset/test_data/test_data/audio_1267.wav'
Error processing /content/dataset/test_data/test_data/audio_683.wav: [Errno 2] No such file or directory: '/content/dataset/test_data/test_data/audio_683.wav'
Error processing /content/dataset/test_data/test_data/audio_1242.wav: [Errno 2] No such file or directory: '/content/dataset/test_data/test_data/audio_1242.wav'
Error process

/tmp/ipython-input-335/45280477.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(audio_path, sr=22050)


In [46]:
# Update this to the correct path found by your search script
TEST_AUDIO_DIR = '/content/sample_data/audio test'

test_features = []

for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    # Construct path using the corrected directory
    audio_path = os.path.join(TEST_AUDIO_DIR, row['filename'])

    if os.path.exists(audio_path):
        try:
            features = extract_audio_features(audio_path)
            if features is not None:
                test_features.append(features)
        except Exception as e:
            print(f"Error processing {audio_path}: {e}")
    else:
        # This will help you verify if the filenames in CSV match the folder
        if i < 5:
            print(f"❌ Still missing: {audio_path}")

 10%|█         | 20/195 [00:00<00:01, 94.49it/s]

Error processing /content/sample_data/audio test/audio_706.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error processing /content/sample_data/audio test/audio_800.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error processing /content/sample_data/audio test/audio_68.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error processing /content/sample_data/audio test/audio_1267.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /u

 16%|█▋        | 32/195 [00:00<00:01, 104.92it/s]

Error processing /content/sample_data/audio test/audio_437.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error processing /content/sample_data/audio test/audio_1217.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error processing /content/sample_data/audio test/audio_831.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error processing /content/sample_data/audio test/audio_1315.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File 

 30%|███       | 59/195 [00:00<00:01, 115.34it/s]

...

Error processing /content/sample_data/audio test/audio_1179.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error processing /content/sample_data/audio test/audio_217.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error processing /content/sample_data/audio test/audio_388.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error processing /content/sample_data/audio test/audio_550.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /

 90%|█████████ | 176/195 [00:01<00:00, 137.07it/s]

Error processing /content/sample_data/audio test/audio_1183.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error processing /content/sample_data/audio test/audio_306.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error processing /content/sample_data/audio test/audio_713.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error processing /content/sample_data/audio test/audio_225.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /

100%|██████████| 195/195 [00:01<00:00, 126.30it/s]

Error processing /content/sample_data/audio test/audio_285.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error processing /content/sample_data/audio test/audio_1178.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error processing /content/sample_data/audio test/audio_135.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /usr/local/lib/python3.12/dist-packages/librosa/core/audio.py, line 33, in <module>

----> resampy = lazy.load("resampy")
Error processing /content/sample_data/audio test/audio_512.wav: No module named 'resampy'

This error is lazily reported, having originally occured in
  File /

In [48]:
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

# Ensure your paths are correct
TEST_AUDIO_DIR = '/content/sample_data/audio test'
test_features = []

# Re-run the loop
for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    audio_path = os.path.join(TEST_AUDIO_DIR, row['filename'])

    if os.path.exists(audio_path):
        try:
            # The error happened here, but resampy is now installed
            audio, sr = librosa.load(audio_path, sr=22050)

            # Extract features (e.g., MFCCs)
            mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
            mfccs_scaled = np.mean(mfccs.T, axis=0)
            test_features.append(mfccs_scaled)

        except Exception as e:
            print(f"Error processing {audio_path}: {e}")
    else:
        if i < 3: print(f"File missing: {audio_path}")

print(f"\n✅ Success! Processed {len(test_features)} test samples.")

100%|██████████| 195/195 [00:36<00:00,  5.41it/s]


✅ Success! Processed 195 test samples.


In [49]:
# Assuming you found your training audio folder earlier
TRAIN_AUDIO_DIR = '/content/sample_data/audio train'

train_features = []
train_labels = []

print("Extracting features from training set...")
for i, row in tqdm(train_df.iterrows(), total=len(train_df)):
    audio_path = os.path.join(TRAIN_AUDIO_DIR, row['filename'])
    if os.path.exists(audio_path):
        try:
            audio, sr = librosa.load(audio_path, sr=22050)
            mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
            train_features.append(np.mean(mfccs.T, axis=0))
            train_labels.append(row['label'])
        except Exception as e:
            continue

# Convert to arrays
X_train = np.array(train_features)
y_train = np.array(train_labels)
X_test = np.array(test_features) # This is the list from your last image

Extracting features from training set...


100%|██████████| 444/444 [01:26<00:00,  5.16it/s]


In [51]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

# We name it 'scoring_model' so it doesn't overwrite your 'model' (Whisper)
scoring_model = Sequential([
    Dense(256, activation='relu', input_shape=(40,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(1)
])

scoring_model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
print("Scoring model initialized!")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Scoring model initialized!


In [52]:
# Train the NEW scoring_model for 20 epochs
history = scoring_model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - loss: 5.7488 - mae: 2.0214 - val_loss: 633.9685 - val_mae: 25.0025
Epoch 2/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 2.5603 - mae: 1.2021 - val_loss: 175.5837 - val_mae: 13.1494
Epoch 3/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.6179 - mae: 1.0379 - val_loss: 71.8896 - val_mae: 8.3645
Epoch 4/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.4044 - mae: 0.9568 - val_loss: 6.8106 - val_mae: 2.3178
Epoch 5/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.2423 - mae: 0.8765 - val_loss: 9.3278 - val_mae: 2.8536
Epoch 6/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1.3333 - mae: 0.9157 - val_loss: 0.8187 - val_mae: 0.7274
Epoch 7/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.2565 - mae: 0.9020 - val_loss: 0.5608 - val_mae: 0.6302
Epoch 8/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.1375 - mae: 0.8751 - val_loss: 0.7295 - val_mae: 0.6821
Epoch 9/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.80

In [53]:
# Use the trained Keras model to predict scores for the test features
# We flatten the results so they fit into a simple column
predictions = scoring_model.predict(X_test).flatten()

print(f"✅ Generated {len(predictions)} predictions.")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
✅ Generated 195 predictions.


In [54]:
# Create the final DataFrame
submission = pd.DataFrame({
    'filename': test_df['filename'].iloc[:195],
    'label': predictions
})

# Optional: Ensure scores stay within the standard 0-5 range
submission['label'] = submission['label'].clip(0, 5)

# Save to the Colab environment
submission.to_csv('submission.csv', index=False)

print("🚀 submission.csv is ready!")

🚀 submission.csv is ready!
